# Federated Learning (FedAvg)
### ResNet18 on CIFAR-10 (Dirichlet α = 0.5)

In [1]:
#=============================================================================
# Federated Training (FedAvg): ResNet18 on CIFAR-100
# Non-IID client splits via Dirichlet (α = 0.5)
# 20 Federated rounds
# LR warmup on first 5 rounds, the cosine
# Logs:
#    (1) acc_train (Weighted across selected clients)
#    (2) acc_test
#    (3) round_time_sec
#    (4) mean_FgT_loss
# ============================================================================

import os
import time    # Measuring wall-clock time per epoch to compare efficiency
import math    # Cosine learning-rate schedule to stabilize training
import random
import copy


# ============================================================================
#                                Third-party
# ============================================================================
import numpy as np
import torch                                                   # Core deep learning tensor library (GPU/CPU compute)
from torch import nn                                           # Neural network layers/losses
import torch.nn.functional as F                                # Functional ops like softmax
from torch.utils.data import DataLoader, Dataset, Subset       # Batching datasets
from torchvision import datasets, transforms, models           # CIFAR datasets + transforms + ResNet
import pandas as pd

# ============================================================================
#                               Matplotlib    
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


# ============================================================================
#                         Colored terminal output
# ============================================================================
# To print in color -------test/train of the client side
def prRed(skk): 
    print("\033[91m {}\033[00m" .format(skk)) 
def prGreen(skk): 
    print("\033[92m {}\033[00m" .format(skk))   


def _sync_cuda():
    """
    Synchronize CUDA to get accurate timing.
    Compare wall time across methods.
    """
    if torch.cuda.is_available():
        torch.cuda.synchronize()


# ============================================================================
#                         Configuration
# ============================================================================
PROGRAM_NAME = "FL - CIFAR-10 α=0.5"
DATASET_NAME = "CIFAR10"            # Set toggle between CIFAR-10 and CIFAR-100
DATA_ROOT = "data"                  # Set where torchvision stores CIFAR data


# ANSI colors
RED   = "\033[91m"
GREEN = "\033[92m"
YEL   = "\033[93m"
CYAN  = "\033[96m"
RESET = "\033[0m"

# ---------------------------- Federated Setup ---------------------------- #
EPOCHS = 200
LOCAL_EPOCHS = 5
NUM_CLIENTS = 10
FRAC = 1.0                         # Fraction of clients per rounds (m = max(1, int(FRAC * NUM_CLIENTS)))
ALPHA = 0.5


# ----------------------------- Optimization ------------------------------ #
BATCH_SIZE = 256
LR_BASE = 0.0001  
WARMUP_ROUNDS = 0
MOMENTUM = 0.9                     # Smoothing/Accelerating SGD undates
WEIGHT_DECAY = 5e-4                # L2 Regularization
LABEL_SMOOTH = 0.0
NUM_WORKERS = 4
SEED = 1234


# Turn off cosine. Only use constant LR 
cosine_schedule = False
if cosine_schedule:
    SCHEDULER = "cosine"               # Making it the same shape across runs
else:
    prRed(f"Using constant LR = {LR_BASE} (cosine_schedule={cosine_schedule})")


# ============================================================================
#                             Reproduce
# ============================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)                            # Seed CPU RNG for Pytorch ops
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)               # Seed all GPUs for DataParallel)
    torch.backends.cudnn.deterministic = True      # Pick deterministic algorithms
    torch.backends.cudnn.benchmark = False         # Turn off autotuner for derterminism 


# ============================================================================
#                           Device Selection
# ============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

print(f"---------{PROGRAM_NAME}----------")  


# ============================================================================
#                           Data & Transforms
#               Setting mean/std between CIFAR-10 & CIFAR-100
#    Normallizing inpits accelerates convergence and stabilizes training
# ============================================================================
CIFAR_MEAN = {
    "CIFAR10":  [0.4914, 0.4822, 0.4465],
    "CIFAR100": [0.5071, 0.4867, 0.4408],
}[DATASET_NAME]
CIFAR_STD = {
    "CIFAR10":  [0.2470, 0.2435, 0.2616],
    "CIFAR100": [0.2675, 0.2565, 0.2761],
}[DATASET_NAME]


# Training time
train_tfms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),   
    transforms.RandomHorizontalFlip(),      
    transforms.ToTensor(),                        # Convert PIL Image -> torch.FloatTensor in [0,1]
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),  # Normalize to zero-mean/unit-var per channel
])

# Test-time we only normalize, no random, to get stable evaluation.
test_tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])


# ============================================================================
#                               DataLoders
# ============================================================================
# Download CIFAR datasets
if DATASET_NAME == "CIFAR10":
    train_set = datasets.CIFAR10(root=DATA_ROOT, train=True,  download=True, transform=train_tfms)
    test_set  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=test_tfms)
    NUM_CLASSES = 10
else:
    train_set = datasets.CIFAR100(root=DATA_ROOT, train=True,  download=True, transform=train_tfms)
    test_set  = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=test_tfms)
    NUM_CLASSES = 100


# Shuffling, batching, perfetching
train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)

test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,  # No shuffle at test time
    num_workers=NUM_WORKERS, pin_memory=True
)

# ============================================================================
#                           Dirichlet Non-IID
# ============================================================================
def dirichlet_noniid_partition(labels: np.ndarray, n_clients: int, alpha: float, min_size_per_client: int = 10):
    """
    Create a label-skewed non-IID split using Dirichlet(alpha).

    Args:
        labels: np.array of shape (N,) with class ids
        n_clients: number of clients
        alpha: Dirichlet concentration (lower = more skew)
        min_size_per_client: ensure no client is empty

    Returns:
        list of index lists; indices for each client
    """
    n_classes = labels.max() + 1
    label_indices = [np.where(labels == y)[0] for y in range(n_classes)]
    for c in range(n_classes):
        np.random.shuffle(label_indices[c])

    client_indices = [[] for _ in range(n_clients)]
    class_proportions = np.random.dirichlet(alpha=[alpha] * n_clients, size=n_classes)

    # Allocate per class following Dirichlet proportions
    for c in range(n_classes):
        idx_c = label_indices[c]
        proportions = class_proportions[c]
        # number of samples of class c to give client k:
        counts = (proportions * len(idx_c)).astype(int)
        # Adjust remainder
        while counts.sum() < len(idx_c):
            counts[np.argmax(proportions)] += 1

        start = 0
        for k in range(n_clients):
            take = counts[k]
            client_indices[k].extend(idx_c[start:start + take].tolist())
            start += take

    # Shuffle per client and enforce min size (if tiny clients exist, merge)
    for k in range(n_clients):
        random.shuffle(client_indices[k])

    # If any client is too small, we’ll borrow from the largest pool
    small = [k for k in range(n_clients) if len(client_indices[k]) < min_size_per_client]
    if len(small) > 0:
        big = sorted(range(n_clients), key=lambda k: len(client_indices[k]), reverse=True)
        for sk in small:
            need = min_size_per_client - len(client_indices[sk])
            for bk in big:
                if bk == sk:
                    continue
                give = min(need, max(0, len(client_indices[bk]) - min_size_per_client))
                if give > 0:
                    moved = client_indices[bk][:give]
                    client_indices[sk].extend(moved)
                    client_indices[bk] = client_indices[bk][give:]
                    need -= give
                if need <= 0:
                    break
        for k in range(n_clients):
            random.shuffle(client_indices[k])

    return client_indices


y_train = np.array(train_set.targets)
client_indices = dirichlet_noniid_partition(y_train, NUM_CLIENTS, ALPHA)

client_loaders = []
for idxs in client_indices:
    subset = Subset(train_set, idxs)
    ld = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    client_loaders.append(ld)

test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


#=============================================================================
#                                Model
#                 ResNet-18 for 32 x 32 CIFAR input:
#                   (1) Conv1: 3 x 3 stride-1
#                   (2) Remove initial maxpool
#============================================================================= 

def build_cifar_resnet18(num_classes: int) -> nn.Module:
    m = models.resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

global_model = build_cifar_resnet18(NUM_CLASSES).to(device)


#=============================================================================
#                                Loss settings
#============================================================================= 
def make_ce():
    # keep label_smoothing consistent across train/eval
    return nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)


#=============================================================================
#                               FL Core
#============================================================================= 

def train_local(model, loader, device, epochs, lr,
                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
                client_id=None, verbose=True):
    """
    Returns (state_dict, num_samples, avg_loss, num_correct)
    Prints per-client, per-local-epoch Acc/Loss when verbose=True.
    """
    local = copy.deepcopy(model).to(device).train()
    opt = torch.optim.SGD(local.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    ce = make_ce()

    total_loss, total_cnt, correct = 0.0, 0, 0

    for e in range(epochs):
        # per-local-epoch accumulators
        ep_loss, ep_cnt, ep_correct = 0.0, 0, 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            logits = local(x)
            loss = ce(logits, y)
            loss.backward()
            opt.step()

            bs = y.size(0)
            total_loss += loss.item() * bs
            total_cnt  += bs
            correct    += (logits.argmax(1) == y).sum().item()

            ep_loss    += loss.item() * bs
            ep_cnt     += bs
            ep_correct += (logits.argmax(1) == y).sum().item()


        # print after each local epoch
        if verbose:
            acc_avg_train  = 100.0 * ep_correct / max(1, ep_cnt)
            loss_avg_train = ep_loss / max(1, ep_cnt)
        
        
            prGreen(" Client{} Train => Local Epoch: {} \tAcc: {:.3f} \tLoss: {:.4f}".format(
                (client_id if client_id is not None else "?"), e, acc_avg_train, loss_avg_train
            ))
        


    avg_loss = total_loss / max(1, total_cnt)
    return copy.deepcopy(local.state_dict()), total_cnt, avg_loss, correct


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval().to(device)
    ce = make_ce()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = ce(logits, y).item()
        loss_sum += loss * y.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        total    += y.size(0)
    return (loss_sum / max(1, total), 100.0 * correct / max(1, total))


def fedavg(state_dicts, sample_counts):
    """
    Weighted parameter average by number of samples.
    """
    total = float(sum(sample_counts))
    avg = copy.deepcopy(state_dicts[0])
    for k in avg.keys():
        avg[k] = sum(sd[k] * (n / total) for sd, n in zip(state_dicts, sample_counts))
    return avg


#=============================================================================
#                           Warmup + Cosine
#                             Per-round LR
#============================================================================= 
def lr_for_round(rnd: int) -> float:
    # linear warmup
    if rnd <= WARMUP_ROUNDS:
        return LR_BASE * rnd / WARMUP_ROUNDS

    
    # cosine decay after warmup
    progress = (rnd - WARMUP_ROUNDS) / max(1, (EPOCHS - WARMUP_ROUNDS))
    return 0.5 * LR_BASE * (1.0 + math.cos(math.pi * progress))


#=============================================================================
#                         Windowed forgetting metric
# Splitting the test set into EPOCHS disjoint windows.
# (1) - After each epoch e, update the best-so-far loss on window e.
# (2) - Measuring how much the loss on all past windows has risen vs there historical minimum.
#============================================================================= 
class DatasetSplit(Dataset):
    def __init__(self, base: Dataset, idxs):
        self.base, self.idxs = base, list(idxs)
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, i: int):
        return self.base[self.idxs[i]]

        
# Build disjoint validation loaders per epoch index (round-robin partitioning).
val_loaders = {}
idx_all = np.arange(len(test_set))
for t in range(EPOCHS):
    idxs = idx_all[t::EPOCHS]            # window t gets items t, t+EPOCHS, t+2*EPOCHS, ...
    val = DatasetSplit(test_set, idxs)
    val_loaders[t] = DataLoader(
        val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
    )


best_loss_per_window = [float("inf")] * EPOCHS  # Track each window's best (min) loss so far


# Evaluate average loss on a particular window.
@torch.no_grad()
def eval_loss_on_window(model: nn.Module, t: int) -> float:
    loader = val_loaders[t]
    if len(loader.dataset) == 0:
        return 0.0
    model.eval()
    ce = make_ce()

    total = 0
    loss_sum = 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss = ce(logits, y).item()

        # Skip non-finite losses to avoid poisoning FgT
        if not math.isfinite(loss):
            continue

        loss_sum += loss * y.size(0)
        total    += y.size(0)

    if total == 0:
        # No valid batches for this window; treat as "no signal"
        return float("nan")

    return loss_sum / total



#=============================================================================
#                         Training loop
#============================================================================= 
acc_train_hist   = []
acc_test_hist    = []
round_time_hist  = []
mean_fgt_hist    = []



for rnd in range(1, EPOCHS + 1):
    _sync_cuda(); t0 = time.perf_counter()

    if cosine_schedule:
        lr = lr_for_round(rnd)
    else:
        lr = LR_BASE

    print(f"[Round {rnd}] lr = {lr:.6f}")
    m = max(1, int(np.ceil(FRAC * NUM_CLIENTS)))
    selected = np.random.choice(NUM_CLIENTS, m, replace=False)

    local_states = []
    local_counts = []
    local_losses = []
    local_corrects = []

    for k in selected:
        prRed(" Client{} => ".format(
            k)
               )
        sd, cnt, loss_k, correct_k = train_local(
            global_model,
            client_loaders[k],
            device,
            epochs=LOCAL_EPOCHS,
            lr=lr,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
            client_id=int(k),     # Shows up in the print line
            verbose=True          # Per-local-epoch lines
        )
        local_states.append(sd)
        local_counts.append(cnt)
        local_losses.append(loss_k)
        local_corrects.append(correct_k)


    # FedAvg aggregation
    new_global = fedavg(local_states, local_counts)
    global_model.load_state_dict(new_global)

    # Evaluate global
    test_loss, test_acc = evaluate(global_model, test_loader, device)

    # Weighted mean train accuracy across selected clients
    total_samples_sel   = max(1, int(sum(local_counts)))
    train_acc_weighted  = (sum(local_corrects) / total_samples_sel) * 100.0
    train_loss_weighted = sum(l * c for l, c in zip(local_losses, local_counts)) / total_samples_sel
    round_label = rnd - 1

    
    _sync_cuda(); dt = time.perf_counter() - t0

    
    # Forgetting metric (use round windows)
    # Current round index as a window id
    t = rnd - 1

    # 1) Update the historical minimum for the *current* window,
    #    but only if the loss is finite.
    L_cur_t = eval_loss_on_window(global_model, t)
    if math.isfinite(L_cur_t):
        if L_cur_t < best_loss_per_window[t]:
            best_loss_per_window[t] = L_cur_t
    # else: skip update; we still have no valid baseline for this window

    # 2) Compute forgetting on all PAST windows (0 .. t-1) vs their historical minima.
    #    Only include windows where BOTH the current loss and the historical best are finite.
    per_win_fgt = []
    for t0 in range(t):
        L_now = eval_loss_on_window(global_model, t0)
        best = best_loss_per_window[t0]

        if not (math.isfinite(L_now) and math.isfinite(best)):
            # No meaningful forgetting number for this window yet
            continue

        Fgt = L_now - best      # allow negatives (no clamp)
        per_win_fgt.append(Fgt)

    mean_fgt = float(np.mean(per_win_fgt)) if per_win_fgt else 0.0



    # Logs
    acc_train_hist.append(train_acc_weighted)
    acc_test_hist.append(test_acc)
    round_time_hist.append(dt)
    mean_fgt_hist.append(mean_fgt)

    
    print(f" Train: \tRound {round_label:3d}, \t\tAcc {train_acc_weighted:0.3f} \t| Loss {train_loss_weighted:0.3f}")
    print(f" Test : \tRound {round_label:3d}, \t\tAcc {test_acc:0.3f}  \t| Loss {test_loss:0.3f}")
    
    # Forgetting (you already computed mean_fgt)
    print(f" Forgetting : Round {round_label}, \tmean_forgetting : {mean_fgt:0.3f}")
    print("=================================================================\n")

print("Training and Evaluation completed!")


#=============================================================================
#                                  Save logs
#============================================================================= 
# Keep lengths aligned (paranoia)
n = min(len(acc_train_hist), len(acc_test_hist), len(round_time_hist), len(mean_fgt_hist))
round_idx = list(range(1, n + 1))

df = pd.DataFrame({
    "round":          round_idx,       
    "acc_train":      acc_train_hist[:n],
    "acc_test":       acc_test_hist[:n],
    "round_time_sec": round_time_hist[:n],
    "mean_FgT_loss":  mean_fgt_hist[:n],
})

csv_name  = f"{PROGRAM_NAME}.csv"     
df.to_csv(csv_name, index=False)    
print("Saved:", csv_name)

#=============================================================================
#                         Program Completed
#============================================================================= 


 Using constant LR = 0.01 (cosine_schedule=False)
NVIDIA H200
---------FL - CIFAR-10 α=0.5----------
[Round 1] lr = 0.010000
  Client2 => 
  Client2 Train => Local Epoch: 0 	Acc: 35.326 	Loss: 1.9742
  Client2 Train => Local Epoch: 1 	Acc: 44.615 	Loss: 1.6420
  Client2 Train => Local Epoch: 2 	Acc: 50.019 	Loss: 1.4584
  Client2 Train => Local Epoch: 3 	Acc: 53.183 	Loss: 1.3332
  Client2 Train => Local Epoch: 4 	Acc: 54.867 	Loss: 1.2853
  Client8 => 
  Client8 Train => Local Epoch: 0 	Acc: 35.576 	Loss: 1.7089
  Client8 Train => Local Epoch: 1 	Acc: 47.364 	Loss: 1.3907
  Client8 Train => Local Epoch: 2 	Acc: 51.046 	Loss: 1.2388
  Client8 Train => Local Epoch: 3 	Acc: 55.735 	Loss: 1.1342
  Client8 Train => Local Epoch: 4 	Acc: 58.119 	Loss: 1.0644
  Client7 => 
  Client7 Train => Local Epoch: 0 	Acc: 48.084 	Loss: 1.5547
  Client7 Train => Local Epoch: 1 	Acc: 60.537 	Loss: 1.2383
  Client7 Train => Local Epoch: 2 	Acc: 62.504 	Loss: 1.1231
  Client7 Train => Local Epoch: 3 	Acc: 